In [30]:
# Import necessary libraries
import os
from pathlib import Path
import numpy as np
import pandas as pd
from astropy import coordinates as coords
from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.table import Table
from astropy.io import fits
from astropy.cosmology import LambdaCDM
from tqdm import tqdm
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

# Set cosmology
cosmo = LambdaCDM(H0=70, Om0=0.3, Ode0=0.7)

plt.rcParams.update({
    "font.family": 'STIXGeneral',
    'text.usetex': False,
    "mathtext.fontset": 'cm',
    "axes.labelweight": "bold",
    'font.size': 25,
    'font.weight': 'normal',
    
    # Tick direction and appearance
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,            # show top ticks
    'ytick.right': True,          # show right ticks
    'xtick.minor.visible': True,  # show minor x ticks
    'ytick.minor.visible': True,  # show minor y ticks
    'xtick.major.size': 10,
    'xtick.minor.size': 6,
    'ytick.major.size': 10,
    'ytick.minor.size': 6,
    'xtick.major.width': 1.6,
    'xtick.minor.width': 1.6,
    'ytick.major.width': 1.6,
    'ytick.minor.width': 1.6,
    
    # Axes and line properties
    'lines.linewidth': 2,
    'axes.linewidth': 3.5,
    'axes.labelpad': 4,
    'xtick.major.pad': 7,
    'image.origin': 'lower'
})
# Pandas configuration
pd.set_option('display.max_columns', None)

In [31]:
df = pd.read_csv('./A2199_mastercat_intermediate_file0.csv')
df['p_modelmag_u_0'] = df['p_modelmag_u'] - df['p_extinction_u']
df['p_modelmag_g_0'] = df['p_modelmag_g'] - df['p_extinction_g']
df['p_modelmag_r_0'] = df['p_modelmag_r'] - df['p_extinction_r']
df['p_modelmag_i_0'] = df['p_modelmag_i'] - df['p_extinction_i']
df['p_modelmag_z_0'] = df['p_modelmag_z'] - df['p_extinction_z']

df['p_petromag_u_0'] = df['p_petromag_u'] - df['p_extinction_u']
df['p_petromag_g_0'] = df['p_petromag_g'] - df['p_extinction_g']
df['p_petromag_r_0'] = df['p_petromag_r'] - df['p_extinction_r']
df['p_petromag_i_0'] = df['p_petromag_i'] - df['p_extinction_i']  
df['p_petromag_z_0'] = df['p_petromag_z'] - df['p_extinction_z']  
df['grmod'] = df['p_modelmag_g_0'] - df['p_modelmag_r_0']

In [32]:
galaxy_check_vis = pd.read_csv('./04d_A2199galaxy_visual_classification_result.csv')
pointsource_flag_vis = pd.read_csv('./05d_point_source_withz_SDSSimglist_result.csv')
petromag_flag_vis = pd.read_csv('./06d_petromag_validity_check_SDSSimglist_result.csv')

# 1. Update photometry put $r_\mathrm{fiber}$ into $r_\mathrm{petro}$ for "vis_phot_update_target" galaxies 

In [33]:
vis_phot_update_target = petromag_flag_vis[petromag_flag_vis['Flag']==0]
vis_phot_update_target.rename(columns={'objid':'p_objid'}, inplace=True)
# Create a set of target objids for fast lookup (lpz)
target_objids = set(vis_phot_update_target['p_objid'])

# Create a mask for rows to update (lpz)
update_mask = df['p_objid'].isin(target_objids)

# For the selected rows, set the petro magnitude to the fiber magnitude (lpz)
df.loc[update_mask, 'p_petromag_r'] = df.loc[update_mask, 'p_fibermag_r']

# Add/overwrite the 'photflag' column: 1 for updated, 0 otherwise (lpz)
df['photflag'] = 0
df.loc[update_mask, 'photflag'] = 1

/var/folders/8j/r5cxq2xj3bxdkw71t64p365m0000gn/T/ipykernel_80617/2648005491.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  vis_phot_update_target.rename(columns={'objid':'p_objid'}, inplace=True)


# 2. Update galaxyflag column in df

In [34]:
# Merge visual galaxy classification into master table
df = df.merge(galaxy_check_vis[['p_objid', 'galaxyflag']], on='p_objid', how='left')

# Remove sources classified as non-galaxy but having measured redshift
remove_mask = (df['galaxyflag'] == 0) & (df['z_tot_z'] != -9)
removed_objids = df.loc[remove_mask, 'p_objid']

print("제거되는 source는 galaxy가 아닌데도 redshift가 있었고, 확인 결과 NED에서 galaxy fragmentation으로 판명되었습니다.")
print(f"제거 대상 개수: {remove_mask.sum()}")
print(removed_objids)

# Fill missing visual classifications as non-galaxy (0)
df['galaxyflag'] = df['galaxyflag'].fillna(0).astype(int)

# Drop those rows and reset index
df = df.loc[~remove_mask].reset_index(drop=True)


제거되는 source는 galaxy가 아닌데도 redshift가 있었고, 확인 결과 NED에서 galaxy fragmentation으로 판명되었습니다.
제거 대상 개수: 1
6784    1237659326566039624
Name: p_objid, dtype: int64


# 3. MMT Redshift Assign 오류 ?

In [35]:
df[df['p_objid']==1237659326566236561]

,phot_source,p_phtype0,p_radgal,p_objid,p_ra,p_dec,p_petromag_u,p_petromagerr_u,p_petromag_g,p_petromagerr_g,p_petromag_r,p_petromagerr_r,p_petromag_i,p_petromagerr_i,p_petromag_z,p_petromagerr_z,p_modelmag_u,p_modelmagerr_u,p_modelmag_g,p_modelmagerr_g,p_modelmag_r,p_modelmagerr_r,p_modelmag_i,p_modelmagerr_i,p_modelmag_z,p_modelmagerr_z,p_fibermag_u,p_fibermagerr_u,p_fibermag_g,p_fibermagerr_g,p_fibermag_r,p_fibermagerr_r,p_fibermag_i,p_fibermagerr_i,p_fibermag_z,p_fibermagerr_z,p_extinction_u,p_extinction_g,p_extinction_r,p_extinction_i,p_extinction_z,p_petrorad_r,p_petroraderr_r,p_devrad_i,p_devab_i,p_run,p_rerun,p_camcol,p_field,p_efac,p_probpsf,z_mmt_xcr,z_mmt_tfilename,z_dfilename,z_filename,z_mmt_z,z_mmt_zerr,z_mmt_velqual,z_ned_name,z_ned_z,z_ned_zerr,z_sdss_z,z_sdss_zerr,z_desi_id,SURVEY,z_desi_z,z_desi_zerr,FLUX_R,FLUX_IVAR_R,z_tot_z,z_tot_zsource,z_tot_zerr,p_modelmag_u_0,p_modelmag_g_0,p_modelmag_r_0,p_modelmag_i_0,p_modelmag_z_0,p_petromag_u_0,p_petromag_g_0,p_petromag_r_0,p_petromag_i_0,p_petromag_z_0,grmod,photflag,galaxyflag
10868,DR9,8,25.151339,1237659326566236561,247.569504,39.823536,20.86823,0.198283,20.51311,0.224491,20.63806,0.321844,20.3703,0.346669,20.57846,0.651303,20.79026,0.076691,20.39696,0.027515,20.52836,0.036063,20.34023,0.046999,20.43827,0.160604,20.31249,0.073684,19.6076,0.056651,19.40737,0.060058,19.1467,0.065085,19.17897,0.106244,0.057319,0.042175,0.030589,0.023194,0.016445,1.293596,0.094087,0.48958,0.452035,3225,301,5,238,1.230688,0,23.04,/Users/hhwang/Research/Work/MMTraw/2019.0429/r...,skysub_a2199a19_1,250.a2199a19_1_1260.ms.fits,0.070268,0.000018,N,NN,-9.0,-9.0,-9.0,-9.0,NN,NaN,-9.0,-9.0,NaN,NaN,0.070268,MMT,0.000018,20.732941,20.354785,20.497771,20.317036,20.421825,20.810911,20.470935,20.607471,20.347106,20.562015,-0.142986,0,1


In [36]:
df[df['p_objid']==1237659326566236559]

,phot_source,p_phtype0,p_radgal,p_objid,p_ra,p_dec,p_petromag_u,p_petromagerr_u,p_petromag_g,p_petromagerr_g,p_petromag_r,p_petromagerr_r,p_petromag_i,p_petromagerr_i,p_petromag_z,p_petromagerr_z,p_modelmag_u,p_modelmagerr_u,p_modelmag_g,p_modelmagerr_g,p_modelmag_r,p_modelmagerr_r,p_modelmag_i,p_modelmagerr_i,p_modelmag_z,p_modelmagerr_z,p_fibermag_u,p_fibermagerr_u,p_fibermag_g,p_fibermagerr_g,p_fibermag_r,p_fibermagerr_r,p_fibermag_i,p_fibermagerr_i,p_fibermag_z,p_fibermagerr_z,p_extinction_u,p_extinction_g,p_extinction_r,p_extinction_i,p_extinction_z,p_petrorad_r,p_petroraderr_r,p_devrad_i,p_devab_i,p_run,p_rerun,p_camcol,p_field,p_efac,p_probpsf,z_mmt_xcr,z_mmt_tfilename,z_dfilename,z_filename,z_mmt_z,z_mmt_zerr,z_mmt_velqual,z_ned_name,z_ned_z,z_ned_zerr,z_sdss_z,z_sdss_zerr,z_desi_id,SURVEY,z_desi_z,z_desi_zerr,FLUX_R,FLUX_IVAR_R,z_tot_z,z_tot_zsource,z_tot_zerr,p_modelmag_u_0,p_modelmag_g_0,p_modelmag_r_0,p_modelmag_i_0,p_modelmag_z_0,p_petromag_u_0,p_petromag_g_0,p_petromag_r_0,p_petromag_i_0,p_petromag_z_0,grmod,photflag,galaxyflag
10881,DR9,9,25.19616,1237659326566236559,247.570895,39.823441,17.85694,0.038351,16.79143,0.035355,16.4286,0.081272,16.29547,0.291583,16.32974,0.477721,17.97785,0.022511,16.78998,0.004393,16.44835,0.004312,16.23527,0.005545,16.18529,0.015291,19.90454,0.033805,18.68609,0.024421,18.25266,0.051256,17.99365,0.163092,17.84956,0.18551,0.057412,0.042243,0.030638,0.023232,0.016472,6.345457,0.149922,6.62471,0.740933,3225,301,5,238,-1.824057,0,-9.0,NN,NN,nn.fits,-9.0,-9.0,N,2MASS J16301697+3949236,0.0701,0.000007,0.0701,0.000007,NN,NaN,-9.0,-9.0,NaN,NaN,0.0701,SDSS,0.000007,17.920438,16.747737,16.417712,16.212038,16.168818,17.799528,16.749187,16.397962,16.272238,16.313268,0.330025,0,1


1237659326566236559와 1237659326566236561는 서로 다른 천체처럼 보이지만, 실제로는 같은 대상의 분할(fragmentation)로 보인다. (1237659326566236559가 중심)


이때 1237659326566236559는 SDSS redshift, 1237659326566236561는 MMT redshift가 들어가 있는데, 두 redshift 값이 거의 동일하므로 MMT redshift 정보가 1237659326566236559에 들어가야 했던 것으로 판단된다.


따라서 1237659326566236561에 잘못 들어간 MMT 관련 정보를 1237659326566236559로 옮겨 반영하고, 중복된 1237659326566236561 행은 제거한다.

In [37]:
# 업데이트할 컬럼들
cols = [
    'z_mmt_xcr', 'z_mmt_tfilename', 'z_dfilename', 'z_filename',
    'z_mmt_z', 'z_mmt_zerr', 'z_mmt_velqual',
    'z_tot_z', 'z_tot_zsource', 'z_tot_zerr'
]

src_id = 1237659326566236561   # 값 가져올 row
dst_id = 1237659326566236559   # 값 덮어쓸 row

# source row가 여러 개일 가능성 대비: 첫 번째 것 사용
src_vals = df.loc[df['p_objid'] == src_id, cols].iloc[0]

# destination row에 값 업데이트
df.loc[df['p_objid'] == dst_id, cols] = src_vals.values

# source row 삭제
df = df[df['p_objid'] != src_id].copy()

# 4. Make Extended Source Flag

In [38]:
pointsource_flag_vis[pointsource_flag_vis['SourceFlag']==0]

,objid,RA,DEC,SourceFlag
141,1237659326566039619,247.163557,40.125913,0
158,1237659326566039629,247.164539,40.120931,0
213,1237659326566236252,247.641373,39.830893,0


In [39]:
pointsource_flag_vis[pointsource_flag_vis['SourceFlag']==2]

,objid,RA,DEC,SourceFlag
66,1237659330315288687,246.929210,39.381253,2
85,1237655471820898598,247.550113,39.553527,2


In [40]:
# Merge SourceFlag from pointsource visual check
point_flags = pointsource_flag_vis[['objid', 'SourceFlag']].rename(columns={'objid': 'p_objid'})
df = df.merge(point_flags, on='p_objid', how='left')

# Remove rows where Flag == 0
remove_mask = df['SourceFlag'] == 0
print(f"Removing rows with SourceFlag == 0: {remove_mask.sum()}")
df = df.loc[~remove_mask].copy()

# set SourceFlag to -9 only for rows where it is NA
df.loc[df['SourceFlag'].isna(), 'SourceFlag'] = -9

df.reset_index(drop=True, inplace=True)

Removing rows with SourceFlag == 0: 3


finalize extended source flag

In [41]:
# 1) base mapping from p_probpsf
#    p_probpsf == 1 -> extended_source_flag = 0
#    p_probpsf == 0 -> extended_source_flag = 1
df["extended_source_flag"] = df["p_probpsf"].map({1: 0, 0: 1})

# 2) override: if SourceFlag == 2, force extended_source_flag = 1
if "SourceFlag" in df.columns:
    override_mask = df["SourceFlag"] == 2
    df.loc[override_mask, "extended_source_flag"] = 1
    print(f"SourceFlag==2 override count: {override_mask.sum()}")

    # 3) drop SourceFlag column
    df.drop(columns=["SourceFlag"], inplace=True)
    print("Dropped column: SourceFlag")
else:
    print("SourceFlag column not found; only p_probpsf mapping applied.")


print(df[["p_objid", "p_probpsf", "extended_source_flag"]].head())

SourceFlag==2 override count: 2
Dropped column: SourceFlag
               p_objid  p_probpsf  extended_source_flag
0  1237659325492298748          0                     1
1  1237659325492298738          1                     0
2  1237659325492298685          0                     1
3  1237659325492298214          0                     1
4  1237659325492298822          1                     0


In [42]:
df.to_csv('A2199_mastercat_intermediate_file1_flag_update.csv', index=False)